<a href="https://colab.research.google.com/github/WaitUps/CourseraExercises-AI-Agents-and-Agentic-AI-Course/blob/main/ProgrammaticPrompting1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Generative AI SDK: Gemini Integration Demo

**Adaptation Note:** This code has been adapted and refined from an earlier version originally designed for the OpenAI model, used as part of an exercise in the following Coursera course and specialization:

*   **Course:** [AI Agents and Agentic AI with Python & Generative AI](https://www.coursera.org/learn/ai-agents-python)
*   **Specialization:** [AI Agent Developer Specialization](https://www.coursera.org/specializations/ai-agents)


This notebook serves as a practical guide and demonstration for interacting with Google's Generative AI models (like Gemini) using the Python SDK within a Colab environment. It covers essential steps and best practices for integrating powerful AI capabilities into your applications.

**Key features and concepts demonstrated:**

*   **API Key Management:** Securely configuring your Google API Key from Colab secrets.
*   **Model Discovery:** Listing available Generative AI models.
*   **Flexible Response Generation:** Implementing a `generate_response` helper function designed to:
    *   Interact seamlessly with various Gemini models.
    *   Handle system instructions (passing them directly or integrating into chat history based on model compatibility).
    *   Format messages correctly for the Gemini API.

The `generate_response` function is a component, built for adaptability and robust handling of model interactions, including detailed examples of how to interpret model outputs.

In [19]:
from google.colab import userdata # Imports the userdata module to access secrets stored in Google Colab
import google.generativeai as genai # Imports the Google Generative AI SDK

# Retrieve the API key from Colab's secrets manager.
# The key should be named 'GOOGLE_API_KEY' in the secrets tab.
api_key = userdata.get('GOOGLE_API_KEY')

# Configure the generative AI library with the retrieved API key.
# This makes the API key available for all subsequent `genai` calls in the notebook.
genai.configure(api_key=api_key)

In [20]:
for m in genai.list_models():
  # Check if the model supports the 'generateContent' method.
  # This is crucial for models that are intended for text generation and conversational AI.
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [21]:
from typing import List, Dict # Used for type hinting to improve code readability and maintainability
import google.generativeai as genai # Imports the Google Generative AI SDK for interacting with Gemini models

def generate_response(messages: List[Dict]) -> str:
    """
    Calls the Gemini LLM to generate a response based on a list of messages.
    It handles system instructions separately if provided as the first message.
    """
    # API key handling:
    # The API key is configured globally via `genai.configure` in a previous cell.
    # This means the API key is set up once for all subsequent API calls with `genai`,
    # so there's no need to explicitly pass it or check environment variables within this function for `genai` models.

    system_instruction_content = ""

    # System Instruction Extraction:
    # The function checks if the first message in the list has the role 'system'.
    # If present, this message is treated as a system instruction to guide the model's behavior.
    if messages and messages[0]["role"] == "system":
        system_instruction_content = messages[0]["content"]
        # The remaining messages (from the second one onwards) form the actual chat history.
        chat_messages = messages[1:]
    else:
        # If no system instruction is provided, all messages are considered part of the chat history.
        chat_messages = messages

    # Message Formatting for Gemini API:
    # The GenerativeModel expects messages in a specific format for `generate_content`:
    # Each message should be a dictionary like `{'role': 'user'/'model', 'parts': [{'text': '...'}]}`.
    # This loop converts the simpler input message format `{'role': 'user'/'model', 'content': '...'}`
    # into the required structure by wrapping the content in a 'parts' list with a 'text' key.
    formatted_chat_messages = []
    for msg in chat_messages:
        # Assuming roles are 'user' or 'model' for standard chat interactions.
        formatted_chat_messages.append({'role': msg['role'], 'parts': [{'text': msg['content']}]})

    # Model Initialization:
    # A GenerativeModel instance is created, specifying the desired model ('gemini-flash-latest').
    # 'gemini-flash-latest' is chosen here as it generally offers a good balance of performance,
    # cost-effectiveness, and supports the `system_instruction` parameter directly, making it suitable
    # for many general-purpose tasks.
    # The extracted `system_instruction_content` is passed directly to the model during initialization.
    gemini_model = genai.GenerativeModel(
        'gemini-flash-latest',
        system_instruction=system_instruction_content # Pass the system instruction here, if any.
    )

    # Content Generation:
    # The `generate_content` method is called with the formatted chat messages.
    # `generation_config` is used to control various aspects of the generation process,
    # such as `max_output_tokens` (limiting the response length), `temperature` (creativity),
    # `top_p`, `top_k`, etc. Here, `max_output_tokens` is set to 1024.
    response = gemini_model.generate_content(
        formatted_chat_messages,
        generation_config=genai.GenerationConfig(max_output_tokens=1024)
    )
    # Return Value:
    # The generated text content from the model's response is extracted and returned.
    return response.text


In [23]:
messages = [
    {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
print(response)

In functional programming, we prioritize immutability and declarative transformations. The most idiomatic way to achieve a key-value swap in Python—while remaining functional—is using a dictionary comprehension or higher-order functions like `map`.

Here is the implementation, including a version that handles potential collisions (where multiple keys have the same value).

### 1. The Standard Swap (One-to-One)
This approach assumes your values are unique. If there are duplicate values, the last key encountered will overwrite the others.

```python
from typing import Dict, TypeVar

K = TypeVar('K')
V = TypeVar('V')

def swap_dict(d: Dict[K, V]) -> Dict[V, K]:
    """Swaps keys and values using a declarative dictionary comprehension."""
    return {v: k for k, v in d.items()}

# Example usage:
# data = {'a': 1, 'b': 2}
# result = swap_dict(data) -> {1: 'a', 2: 'b'}
```

### 2. The Higher-Order Function Approach
If you want to avoid comprehension syntax to be even more strictly functional

In [ ]:
response = "#Sample of Generated Response:\n" + response
print(response)

# Sample of Generated Response:

*Note: This is a sample response generated by the AI model. Each execution of the `generate_response` function may produce a different output.*

<br>

---
<br>

In functional programming, we treat data transformations as a series of applications rather than a sequence of imperative steps.

When swapping keys and values, we must consider a critical edge case: **collisions**. Since dictionary keys must be unique but values do not, a simple swap might result in data loss if two keys share the same value.

Below are three ways to approach this, ranging from the "Pythonic" functional style to a more robust "grouping" approach.

### 1. The Declarative Approach (Dictionary Comprehension)
This is the most common functional pattern in Python. It is declarative, as it describes *what* the result should look like rather than *how* to loop through it.

```python
def swap_dict(d: dict) -> dict:
    return {v: k for k, v in d.items()}
```
*Note: If duplicate values exist, the last key encountered will overwrite previous ones.*

---

### 2. The Higher-Order Function Approach
If you prefer using point-free style or composition, you can use `map` and `reversed`. In this context, `reversed` acts on the key-value tuple.

```python
def swap_functional(d: dict) -> dict:
    # d.items() yields (key, value)
    # reversed turns (key, value) into (value, key)
    return dict(map(reversed, d.items()))
```

---

### 3. The Robust Approach (Handling Collisions)
A truly functional approach often avoids data loss. If multiple keys map to the same value, we should "fold" (reduce) the dictionary into a new one where values map to a *list* of keys.

We can achieve this using `functools.reduce`, which is the standard functional "fold" operation.

```python
from functools import reduce

def swap_and_group(d: dict) -> dict:
    def accumulate(acc, item):
        key, value = item
        # Return a new dict (or mutate local acc for performance)
        acc.setdefault(value, []).append(key)
        return acc

    return reduce(accumulate, d.items(), {})

# Example Usage:
data = {'a': 1, 'b': 2, 'c': 1}
print(swap_and_group